# OpenFIGI pipeline

Adds OpenFIGI identifiers to `clean_up_1.asset` (built by `2_f13_securities_pipeline.ipynb`). Every block shows
the database feedback and is written to the run log `logs/openfigi/<RUN_ID>/`. Execution code lives in
`src/pipeline_helpers.py` (`OpenFigiMapper`).

* **US CUSIPs** (first character a digit) are looked up as `ID_CUSIP`
* **Foreign issuers** (CINS, first character a letter, e.g. `G0129K104`) are looked up as `ID_CINS`

The API answers are kept in two tables that are **not** rebuilt with the asset table, so the (slow) lookup
runs once and section 5 can re-attach the FIGIs after every rebuild of `clean_up_1.asset`.

| Section | Phase label | What happens |
|---|---|---|
| 0.2 Database connection | `connection` | read-only check, then working connection |
| 1. Lookup tables | `figi_tables` | `input_1.openfigi_lookup`, `input_1.openfigi_record` |
| 2. What to look up | `figi_pending` | underlying CUSIPs without a lookup yet |
| 3. OpenFIGI lookup | `figi_lookup` | API calls, resumable |
| 4. Lookup results | `figi_results` | found / not found, security types |
| 5. Attach to asset | `figi_attach` | FIGI columns on `clean_up_1.asset` |
| 6. Checks | `figi_checks` | coverage, name comparison, CUSIPs sharing a share class |

## 0. Setup

### 0.1 Configuration

* `OPENFIGI_API_KEY` (environment variable): optional but strongly recommended. Without a key: 10 CUSIPs per
  request and 25 requests per minute (~2.7 h for ~40,000 CUSIPs); with a free key
  (https://www.openfigi.com/user/signup): 100 per request, 25 per 6 seconds (~2 min).
* `MAX_CUSIPS`: limit for a test run (`None` = everything pending)
* `RETRY_ERRORS`: also re-send CUSIPs whose previous lookup ended with an API error

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent                       # .../Database/Initialisation (notebooks run from notebooks/)
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "config")]

from pipeline_helpers import (OpenFigiMapper, PipelineRun, compare_runs, expected_mismatches,
                              load_logs, phase_durations, steps_table)
from local_config import DSN, OPENFIGI_API_KEY   # machine-specific, git-ignored (template: local_config.example.py)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

LOG_ROOT = ROOT / "logs" / "openfigi"

ASSET_TABLE  = "clean_up_1.asset"
LOOKUP_TABLE = "input_1.openfigi_lookup"       # one row per CUSIP sent to OpenFIGI
RECORD_TABLE = "input_1.openfigi_record"       # composite-level FIGI records per CUSIP

# OPENFIGI_API_KEY is never written to the log

MAX_CUSIPS   = None    # e.g. 50 for a test run
RETRY_ERRORS = True

p = PipelineRun(LOG_ROOT, dsn=DSN, max_rows=1000, exact_counts=True)
print("RUN_ID :", p.run_id)
print("API key:", "set" if OPENFIGI_API_KEY else "not set (slow: 10 CUSIPs per request, 25 requests/min)")

RUN_ID : 20261011_201631
API key: not set (slow: 10 CUSIPs per request, 25 requests/min)


### 0.2 Database connection

Read-only check first (separate connection, `READ ONLY` transaction, rolled back); the working connection
(autocommit) is opened only if the check passes.

In [2]:
CONNECTION_OK = p.check_connection(DSN, schemas=["input_1", "clean_up_1"],
                                   tables=[ASSET_TABLE, LOOKUP_TABLE, RECORD_TABLE])

✅ **[connection] connection_check_server**: `ok` | 0.0s

> READ ONLY transaction, rolled back. Nothing committed.

,server_version,database,db_user,is_standby,txn_read_only,db_size,other_sessions
0,PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04....,Holdings_v2,postgres,False,on,53 GB,2


✅ **[connection] connection_check_schemas**: `ok`

,schema_name,schema_exists,can_create,can_use
0,input_1,True,True,True
1,clean_up_1,True,True,True


✅ **[connection] connection_check_tables**: `ok`

> Existing tables will be dropped and recreated by the pipeline.

,table_name,exists,unlogged,est_rows,total_size
0,clean_up_1.asset,True,False,40593,5304 kB
1,input_1.openfigi_lookup,True,False,39240,4664 kB
2,input_1.openfigi_record,True,False,91764,19 MB


In [3]:
assert CONNECTION_OK, "Connection check failed, see connection_check_server above"
p.connect()

Connected: user=postgres password=xxx dbname=Holdings_v2 host=localhost port=5432


## 1. Lookup tables

* `input_1.openfigi_lookup`: one row per CUSIP sent, with `status` **found** / **not_found** (OpenFIGI does not
  know the identifier) / **error**, the API message and the number of records returned
* `input_1.openfigi_record`: one record per **composite** (country, e.g. the US composite of Apple), not one per
  trading venue (Apple alone has ~300 venue records). When OpenFIGI returns only venue records, the first venue
  record of the composite is kept

`CREATE TABLE IF NOT EXISTS`: re-running keeps earlier answers. Set `RESET_LOOKUP = True` to drop them and
look everything up again.

In [4]:
RESET_LOOKUP = False

tab = p.stage("figi_tables")

if RESET_LOOKUP:
    tab.run("drop_lookup_tables", f"DROP TABLE IF EXISTS {RECORD_TABLE}, {LOOKUP_TABLE};")

tab.run("create_lookup_table", f"""
    CREATE TABLE IF NOT EXISTS {LOOKUP_TABLE} (
        cusip        VARCHAR(9)  PRIMARY KEY,
        id_type      TEXT        NOT NULL CHECK (id_type IN ('ID_CUSIP', 'ID_CINS')),
        status       TEXT        NOT NULL CHECK (status IN ('found', 'not_found', 'error')),
        message      TEXT,
        api_records  INTEGER     NOT NULL,
        fetched_at   TIMESTAMPTZ NOT NULL DEFAULT now()
    );
""")
tab.run("create_record_table", f"""
    CREATE TABLE IF NOT EXISTS {RECORD_TABLE} (
        cusip                VARCHAR(9) NOT NULL REFERENCES {LOOKUP_TABLE} (cusip) ON DELETE CASCADE,
        figi                 CHAR(12)   NOT NULL,
        composite_figi       CHAR(12),
        share_class_figi     CHAR(12),
        name                 TEXT,
        ticker               TEXT,
        exch_code            TEXT,
        market_sector        TEXT,
        security_type        TEXT,
        security_type2       TEXT,
        security_description TEXT,
        PRIMARY KEY (cusip, figi)
    );
""")
tab.run("idx_record_share_class", f"""
    CREATE INDEX IF NOT EXISTS openfigi_record_share_class_idx ON {RECORD_TABLE} (share_class_figi);
""")

✅ **[figi_tables] create_lookup_table**: `ok` | `CREATE TABLE` | 0.0s

> NOTICES: NOTICE:  relation "openfigi_lookup" already exists, skipping

✅ **[figi_tables] create_record_table**: `ok` | `CREATE TABLE` | 0.0s

> NOTICES: NOTICE:  relation "openfigi_record" already exists, skipping

✅ **[figi_tables] idx_record_share_class**: `ok` | `CREATE INDEX` | 0.0s

> NOTICES: NOTICE:  relation "openfigi_record_share_class_idx" already exists, skipping

""


## 2. What to look up

Every asset CUSIP is sent. Pending = no lookup yet (or a previous API error when `RETRY_ERRORS`).

In [5]:
pend = p.stage("figi_pending")

pend.run("asset_by_id_type", f"""
    SELECT CASE WHEN left(a.cusip, 1) ~ '[0-9]' THEN 'ID_CUSIP' ELSE 'ID_CINS' END AS id_type,
           count(*)                                         AS assets,
           count(l.cusip)                                   AS looked_up,
           count(*) FILTER (WHERE l.status = 'found')       AS found,
           count(*) FILTER (WHERE l.status = 'not_found')   AS not_found,
           count(*) FILTER (WHERE l.status = 'error')       AS error,
           count(*) FILTER (WHERE l.cusip IS NULL)          AS never_looked_up
    FROM {ASSET_TABLE} a
    LEFT JOIN {LOOKUP_TABLE} l USING (cusip)
    GROUP BY 1
    ORDER BY 1 DESC;
""", note="Asset table at writing: ID_CUSIP 38,683 | ID_CINS 1,910 underlying CUSIPs")

✅ **[figi_pending] asset_by_id_type**: `ok` | `SELECT 2` | rows: **2** | 0.0s

> Asset table at writing: ID_CUSIP 38,683 | ID_CINS 1,910 underlying CUSIPs

,id_type,assets,looked_up,found,not_found,error,never_looked_up
0,ID_CUSIP,38683,38683,12593,26090,0,0
1,ID_CINS,1910,1910,619,1291,0,0


,id_type,assets,looked_up,found,not_found,error,never_looked_up
0,ID_CUSIP,38683,38683,12593,26090,0,0
1,ID_CINS,1910,1910,619,1291,0,0


In [6]:
pending = pend.run("pending_cusips", f"""
    SELECT a.cusip
    FROM {ASSET_TABLE} a
    LEFT JOIN {LOOKUP_TABLE} l USING (cusip)
    WHERE (l.cusip IS NULL {"OR l.status = 'error'" if RETRY_ERRORS else ""})
    ORDER BY a.cusip;
""", fetch_all=True)

✅ **[figi_pending] pending_cusips**: `ok` | `SELECT 0` | rows: **0** | 0.0s

## 3. OpenFIGI lookup

One request = one transaction, so every answer received is saved even if the run stops. **Interrupt the
kernel (■)** to stop; re-run sections 2–3 to continue with what is still pending. HTTP 429 (rate limit) waits
for the reset and 5xx errors are retried automatically.

In [7]:
mapper = OpenFigiMapper(p.stage("figi_lookup"), LOOKUP_TABLE, RECORD_TABLE, api_key=OPENFIGI_API_KEY)

to_send = pending["cusip"].tolist()[:MAX_CUSIPS]
n_req, secs = mapper.estimate(len(to_send))
print(f"{len(to_send):,} CUSIPs -> {n_req:,} requests, about {secs / 60:,.0f} min at the rate limit")

0 CUSIPs -> 0 requests, about 0 min at the rate limit


In [8]:
looked_up = mapper.run(to_send)

<THESIS_DIR>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OpenFIGI: 0request [00:00, ?request/s]


✅ **[figi_lookup] openfigi_lookup**: `ok` | rows: **0** | 0.0s

> 0 of 0 CUSIPs looked up | composite records stored: 0 | without API key

## 4. Lookup results

In [9]:
res = p.stage("figi_results")

res.run("lookup_status", f"""
    SELECT id_type, status, coalesce(message, '') AS message, count(*) AS cusips,
           round(avg(api_records), 1) AS avg_api_records
    FROM {LOOKUP_TABLE}
    GROUP BY 1, 2, 3
    ORDER BY 1 DESC, 2;
""")

✅ **[figi_results] lookup_status**: `ok` | `SELECT 4` | rows: **4** | 0.0s

,id_type,status,message,cusips,avg_api_records
0,ID_CUSIP,found,,12593,31.4
1,ID_CUSIP,not_found,No identifier found.,26090,0.0
2,ID_CINS,found,,619,35.0
3,ID_CINS,not_found,No identifier found.,1291,0.0


,id_type,status,message,cusips,avg_api_records
0,ID_CUSIP,found,,12593,31.4
1,ID_CUSIP,not_found,No identifier found.,26090,0.0
2,ID_CINS,found,,619,35.0
3,ID_CINS,not_found,No identifier found.,1291,0.0


In [10]:
res.run("record_security_types", f"""
    SELECT market_sector, security_type2, count(DISTINCT cusip) AS cusips, count(*) AS composite_records
    FROM {RECORD_TABLE}
    GROUP BY 1, 2
    ORDER BY cusips DESC
    LIMIT 40;
""")

✅ **[figi_results] record_security_types**: `ok` | `SELECT 12` | rows: **12** | 0.1s

,market_sector,security_type2,cusips,composite_records
0,Equity,Common Stock,4890,66268
1,Corp,Corp,4152,4152
2,Equity,Mutual Fund,2505,16814
3,Pfd,Preferred Stock,836,836
4,Equity,Depositary Receipt,428,4810
5,Equity,REIT,214,2897
6,Equity,Partnership Shares,78,341
7,Equity,Unit,67,90
8,Equity,Warrant,44,44
9,Equity,Right,8,13


,market_sector,security_type2,cusips,composite_records
0,Equity,Common Stock,4890,66268
1,Corp,Corp,4152,4152
2,Equity,Mutual Fund,2505,16814
3,Pfd,Preferred Stock,836,836
4,Equity,Depositary Receipt,428,4810
5,Equity,REIT,214,2897
6,Equity,Partnership Shares,78,341
7,Equity,Unit,67,90
8,Equity,Warrant,44,44
9,Equity,Right,8,13


## 5. Attach to the asset table

For each CUSIP one composite record is chosen: the **US composite** (`exch_code = 'US'`) when there is one, else
a US venue record (`UN`, `UQ`, `UV`, … when OpenFIGI returned only venue records), otherwise the first composite by
exchange code (foreign issuers often have no US listing). `figi` is always the composite FIGI. Columns added
to `clean_up_1.asset`:

* `figi` (the chosen composite FIGI), `share_class_figi` (global, same for every market), `figi_ticker`,
  `figi_name`, `figi_exch_code`, `market_sector`, `security_type2`
* `figi_status`: found / not_found / error, `NULL` = not looked up

The asset table is dropped and rebuilt by `2_f13_securities_pipeline.ipynb`; re-run this section afterwards
(no API calls needed).

In [11]:
att = p.stage("figi_attach")

In [12]:
att.run("add_figi_columns", f"""
    ALTER TABLE {ASSET_TABLE}
        ADD COLUMN IF NOT EXISTS figi             CHAR(12),
        ADD COLUMN IF NOT EXISTS share_class_figi CHAR(12),
        ADD COLUMN IF NOT EXISTS figi_ticker      TEXT,
        ADD COLUMN IF NOT EXISTS figi_name        TEXT,
        ADD COLUMN IF NOT EXISTS figi_exch_code   TEXT,
        ADD COLUMN IF NOT EXISTS market_sector    TEXT,
        ADD COLUMN IF NOT EXISTS security_type2   TEXT,
        ADD COLUMN IF NOT EXISTS figi_status      TEXT;
""")

✅ **[figi_attach] add_figi_columns**: `ok` | `ALTER TABLE` | 0.0s

""


In [13]:
att.run("set_figi", f"""
    WITH pick AS (
        SELECT DISTINCT ON (cusip) *
        FROM {RECORD_TABLE}
        ORDER BY cusip,
                 (exch_code = 'US') DESC NULLS LAST,          -- US composite
                 (exch_code ~ '^U[A-Z]$') DESC NULLS LAST,    -- else a US venue (UN, UQ, UV, ...) of the US composite
                 (figi = composite_figi) DESC NULLS LAST,
                 exch_code, figi
    )
    UPDATE {ASSET_TABLE} a
    SET figi             = coalesce(p.composite_figi, p.figi),
        share_class_figi = p.share_class_figi,
        figi_ticker      = p.ticker,
        figi_name        = p.name,
        figi_exch_code   = p.exch_code,
        market_sector    = p.market_sector,
        security_type2   = p.security_type2
    FROM pick p
    WHERE a.cusip = p.cusip;
""")

✅ **[figi_attach] set_figi**: `ok` | `UPDATE 13212` | rows: **13,212** | 0.2s

""


In [14]:
att.run("set_figi_status", f"""
    UPDATE {ASSET_TABLE} a
    SET figi_status = l.status
    FROM {LOOKUP_TABLE} l
    WHERE a.cusip = l.cusip;
""")
att.run("idx_asset_figi", f"CREATE INDEX IF NOT EXISTS asset_share_class_figi_idx ON {ASSET_TABLE} (share_class_figi);")

✅ **[figi_attach] set_figi_status**: `ok` | `UPDATE 40593` | rows: **40,593** | 0.3s

✅ **[figi_attach] idx_asset_figi**: `ok` | `CREATE INDEX` | 0.0s

""


## 6. Checks

In [15]:
chk = p.stage("figi_checks")

chk.run("figi_coverage", f"""
    SELECT CASE WHEN left(cusip, 1) ~ '[0-9]' THEN 'ID_CUSIP' ELSE 'ID_CINS' END AS id_type,
           CASE WHEN last_year >= 2015 THEN 'listed 2015+' ELSE 'last listed before 2015' END AS period,
           count(*)                                       AS assets,
           count(figi)                                    AS with_figi,
           round(100.0 * count(figi) / count(*), 1)       AS pct
    FROM {ASSET_TABLE}
    GROUP BY 1, 2
    ORDER BY 1 DESC, 2;
""", note="older CUSIPs (delisted securities) are more likely to be unknown to OpenFIGI")

✅ **[figi_checks] figi_coverage**: `ok` | `SELECT 4` | rows: **4** | 0.0s

> older CUSIPs (delisted securities) are more likely to be unknown to OpenFIGI

,id_type,period,assets,with_figi,pct
0,ID_CUSIP,last listed before 2015,23905,3821,16.0
1,ID_CUSIP,listed 2015+,14778,8772,59.4
2,ID_CINS,last listed before 2015,658,50,7.6
3,ID_CINS,listed 2015+,1252,569,45.4


,id_type,period,assets,with_figi,pct
0,ID_CUSIP,last listed before 2015,23905,3821,16.0
1,ID_CUSIP,listed 2015+,14778,8772,59.4
2,ID_CINS,last listed before 2015,658,50,7.6
3,ID_CINS,listed 2015+,1252,569,45.4


### Name comparison

The SEC list name and the OpenFIGI name are written differently (abbreviations, 30-character limit), so this is
only a spot check: rows where the first word differs are the ones worth a look.

In [16]:
chk.run("figi_name_mismatch", f"""
    SELECT cusip, issuer_name, figi_name, figi_ticker, figi_exch_code, security_type2
    FROM {ASSET_TABLE}
    WHERE figi IS NOT NULL
      AND split_part(upper(issuer_name), ' ', 1) <> split_part(upper(figi_name), ' ', 1)
    ORDER BY cusip
    LIMIT 100;
""")

✅ **[figi_checks] figi_name_mismatch**: `ok` | `SELECT 100` | rows: **100** | 0.0s

,cusip,issuer_name,figi_name,figi_ticker,figi_exch_code,security_type2
0,000886AB7,A D C TELECOMMUNICATIONS,ADC TELECOMMUNICATIONS I,TEL F 06/15/13,NOT LISTED,Corp
1,000886AD3,A D C TELECOMMUNICATIONS,ADC TELECOMMUNICATIONS I,TEL 1 06/15/08,TRACE,Corp
2,000886AE1,A D C TELECOMMUNICATIONS,ADC TELECOMMUNICATIONS I,TEL 3.5 07/15/15,NOT LISTED,Corp
3,000886AF8,A D C TELECOMMUNICATIONS,COMMSCOPE CONNECTIVITY,TEL 3.5 07/15/17,NOT LISTED,Corp
4,000945AC9,ADT OPERATIONS,TYCO INTERNATIONAL LTD,TYC 0 07/06/10,TRACE,Corp
...,...,...,...,...,...,...
95,023135106,AMAZON COM INC,AMAZON.COM INC,AMZN,US,Common Stock
96,023135AE6,AMAZON COM INC,AMAZON.COM INC,AMZN 4.75 02/01/09 AI,NOT LISTED,Corp
97,023135AF3,AMAZON COM INC,AMAZON.COM INC,AMZN 4.75 02/01/09,TRACE,Corp
98,023139884,AMBAC FINL GROUP INC,OCTAVE SPECIALTY GROUP INC,OSG,US,Common Stock


,cusip,issuer_name,figi_name,figi_ticker,figi_exch_code,security_type2
0,000886AB7,A D C TELECOMMUNICATIONS,ADC TELECOMMUNICATIONS I,TEL F 06/15/13,NOT LISTED,Corp
1,000886AD3,A D C TELECOMMUNICATIONS,ADC TELECOMMUNICATIONS I,TEL 1 06/15/08,TRACE,Corp
2,000886AE1,A D C TELECOMMUNICATIONS,ADC TELECOMMUNICATIONS I,TEL 3.5 07/15/15,NOT LISTED,Corp
3,000886AF8,A D C TELECOMMUNICATIONS,COMMSCOPE CONNECTIVITY,TEL 3.5 07/15/17,NOT LISTED,Corp
4,000945AC9,ADT OPERATIONS,TYCO INTERNATIONAL LTD,TYC 0 07/06/10,TRACE,Corp
...,...,...,...,...,...,...
95,023135106,AMAZON COM INC,AMAZON.COM INC,AMZN,US,Common Stock
96,023135AE6,AMAZON COM INC,AMAZON.COM INC,AMZN 4.75 02/01/09 AI,NOT LISTED,Corp
97,023135AF3,AMAZON COM INC,AMAZON.COM INC,AMZN 4.75 02/01/09,TRACE,Corp
98,023139884,AMBAC FINL GROUP INC,OCTAVE SPECIALTY GROUP INC,OSG,US,Common Stock


### CUSIPs sharing a share class

Several CUSIPs with the same `share_class_figi` usually belong to one security whose CUSIP changed (e.g. a
rebranding or reorganisation). This can link the old and new CUSIP.

In [17]:
chk.run("shared_share_class", f"""
    SELECT share_class_figi, count(*) AS cusips,
           string_agg(cusip || ' ' || first_year || 'Q' || first_quarter || '-' || last_year || 'Q' || last_quarter,
                      ', ' ORDER BY first_year, first_quarter) AS cusip_periods,
           max(figi_name) AS figi_name
    FROM {ASSET_TABLE}
    WHERE share_class_figi IS NOT NULL
    GROUP BY share_class_figi
    HAVING count(*) > 1
    ORDER BY cusips DESC, share_class_figi
    LIMIT 100;
""")

✅ **[figi_checks] shared_share_class**: `ok` | `SELECT 100` | rows: **100** | 0.0s

,share_class_figi,cusips,cusip_periods,figi_name
0,BBG001S6X756,3,"09069N108 2005Q1-2019Q3, 68404L102 2019Q3-2020...",OPTION CARE HEALTH INC
1,BBG001S7ND43,3,"300487105 2014Q4-2019Q3, 452465107 2019Q3-2019...",LEGACY IMBDS INC
2,BBG001SGYST1,3,"57060U191 2012Q1-2016Q2, 92189F718 2016Q2-2020...",VANECK VECTORS OIL SERVICES
3,BBG001T0PYR4,3,"G2113X134 2016Q2-2020Q3, G2113X159 2020Q3-2020...",CHINA CERAMICS CO LTD
4,BBG001TC5M87,3,"30711Y102 2016Q3-2019Q3, 30711Y201 2019Q3-2020...",FANG HOLDINGS LTD - ADR
...,...,...,...,...
95,BBG001SN60Z1,2,"73935X849 2005Q3-2018Q2, 46137V753 2018Q2-2020Q4",INVESCO FOOD & BEVERAGE ETF
96,BBG001SN6GS4,2,"73935X781 2005Q3-2018Q2, 46137V647 2018Q2-2020Q4",INVESCO SEMICONDUCTORS ETF
97,BBG001SNLCD2,2,"30233G100 2018Q2-2020Q4, 30233G209 2020Q4-2020Q4",EYEPOINT PHARMACEUTICALS INC
98,BBG001SNN4P5,2,"48273U102 2007Q4-2020Q4, 86333M108 2020Q4-2020Q4",STRIDE INC


,share_class_figi,cusips,cusip_periods,figi_name
0,BBG001S6X756,3,"09069N108 2005Q1-2019Q3, 68404L102 2019Q3-2020...",OPTION CARE HEALTH INC
1,BBG001S7ND43,3,"300487105 2014Q4-2019Q3, 452465107 2019Q3-2019...",LEGACY IMBDS INC
2,BBG001SGYST1,3,"57060U191 2012Q1-2016Q2, 92189F718 2016Q2-2020...",VANECK VECTORS OIL SERVICES
3,BBG001T0PYR4,3,"G2113X134 2016Q2-2020Q3, G2113X159 2020Q3-2020...",CHINA CERAMICS CO LTD
4,BBG001TC5M87,3,"30711Y102 2016Q3-2019Q3, 30711Y201 2019Q3-2020...",FANG HOLDINGS LTD - ADR
...,...,...,...,...
95,BBG001SN60Z1,2,"73935X849 2005Q3-2018Q2, 46137V753 2018Q2-2020Q4",INVESCO FOOD & BEVERAGE ETF
96,BBG001SN6GS4,2,"73935X781 2005Q3-2018Q2, 46137V647 2018Q2-2020Q4",INVESCO SEMICONDUCTORS ETF
97,BBG001SNLCD2,2,"30233G100 2018Q2-2020Q4, 30233G209 2020Q4-2020Q4",EYEPOINT PHARMACEUTICALS INC
98,BBG001SNN4P5,2,"48273U102 2007Q4-2020Q4, 86333M108 2020Q4-2020Q4",STRIDE INC


In [18]:
chk.run("holdings_figi_coverage", f"""
    SELECT count(*)                                                           AS holdings_rows,
           count(a.figi)                                                      AS rows_with_figi,
           round(100.0 * count(a.figi) / count(*), 2)                         AS rows_pct,
           round(100.0 * sum(h.value) FILTER (WHERE a.figi IS NOT NULL)
                        / sum(h.value), 2)                                    AS value_pct,
           count(a.figi)                                                      AS rows_with_ticker,
           round(100.0 * count(a.figi) / count(*), 2)                         AS rows_ticker_pct,
           round(100.0 * sum(h.value) FILTER (WHERE a.figi_ticker IS NOT NULL)
                        / sum(h.value), 2)                                    AS ticker_value_pct
    FROM clean_up_1.holdings_clean h
    LEFT JOIN {ASSET_TABLE} a ON a.cusip = h.cusip9;
""", note="exact 9-character CUSIP match; holdings rows whose CUSIP is not in the asset table count as 'no FIGI'")


✅ **[figi_checks] holdings_figi_coverage**: `ok` | `SELECT 1` | rows: **1** | 80.3s

> exact 9-character CUSIP match; holdings rows whose CUSIP is not in the asset table count as 'no FIGI'

,holdings_rows,rows_with_figi,rows_pct,value_pct,rows_with_ticker,rows_ticker_pct,ticker_value_pct
0,101814870,68916548,67.69,78.92,68916548,67.69,78.92


,holdings_rows,rows_with_figi,rows_pct,value_pct,rows_with_ticker,rows_ticker_pct,ticker_value_pct
0,101814870,68916548,67.69,78.92,68916548,67.69,78.92


## 7. Run log analytics

Reads `steps.jsonl` from **all** runs in `logs/openfigi/`.

In [19]:
logs = load_logs(LOG_ROOT)
steps_table(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
31,1,connection,connection_check_server,ok,None,NaN,NaN,NaN,0.021
32,2,connection,connection_check_schemas,ok,None,NaN,NaN,NaN,NaN
33,3,connection,connection_check_tables,ok,None,NaN,NaN,NaN,NaN
34,4,figi_tables,create_lookup_table,ok,CREATE TABLE,-1.0,NaN,NaN,0.001
35,5,figi_tables,create_record_table,ok,CREATE TABLE,-1.0,NaN,NaN,0.000
36,6,figi_tables,idx_record_share_class,ok,CREATE INDEX,-1.0,NaN,NaN,0.000
37,7,figi_pending,asset_by_id_type,ok,SELECT 2,2.0,NaN,NaN,0.035
38,8,figi_pending,pending_cusips,ok,SELECT 0,0.0,NaN,NaN,0.013
39,9,figi_lookup,openfigi_lookup,ok,None,0.0,NaN,NaN,0.004
40,10,figi_results,lookup_status,ok,SELECT 4,4.0,NaN,NaN,0.006


In [20]:
phase_durations(logs, p.run_id)

,seconds
phase,
connection,0.021
figi_tables,0.001
figi_pending,0.048
figi_lookup,0.004
figi_results,0.146
figi_attach,0.554
figi_checks,80.322


In [21]:
compare_runs(logs)

rowcount                      duration_s                
run_id                               20261009_124512 20261011_201631 20261009_124512 20261011_201631
phase        step                                                                                   
connection   connection_check_server             NaN             NaN           0.023           0.021
figi_tables  create_lookup_table                -1.0            -1.0           0.018           0.001
             create_record_table                -1.0            -1.0           0.010           0.000
             idx_record_share_class             -1.0            -1.0           0.003           0.000
figi_pending asset_by_id_type                    2.0             2.0           0.021           0.035
             pending_cusips                  40593.0             0.0           0.042           0.013
figi_lookup  openfigi_lookup                 40593.0             0.0       10001.269           0.004
figi_results lookup_status                       4.0             4.0           0.009           0.006
             record_security_types              12.0            12.0           0.082           0.140
figi_attach  add_figi_columns                   -1.0            -1.0           0.005           0.005
             set_figi                        13212.0         13212.0           0.244           0.233
             set_figi_status                 40593.0         40593.0           0.294           0.294
             idx_asset_figi                     -1.0            -1.0           0.026           0.022
figi_checks  figi_coverage                       4.0             4.0           0.017           0.010
             figi_name_mismatch                100.0           100.0           0.008           0.003
             shared_share_class                100.0           100.0           0.026           0.015
             holdings_figi_coverage              1.0             1.0          84.541          80.294
             commit                             -1.0             NaN           0.000             NaN

In [22]:
p.close()

Run log: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/pipelines/logs/openfigi/20261011_201631/steps.jsonl
